# Análisis e Exploración de Datos de Airbnb Barcelona

Este notebook realiza el análisis exploratorio, la detección y filtrado de duplicados, la investigación de anuncios sin reseñas y la auditoría de licencias turísticas en los anuncios de Airbnb extraídos de **Inside Airbnb** (`insideairbnb_barcelona_2026-06-24_listings.csv`).

### Estructura del Notebook:
1. **Carga e inspección inicial**: total de registros.
2. **Depuración de duplicados por contenido**: eliminación de 68 anuncios repetidos por el mismo anfitrión.
3. **Análisis de anfitriones y concentración de propiedad**: monopropiedad vs multipropiedad.
4. **Tipos de alojamiento** (`room_type`) y su frecuencia.
5. **Concentración geográfica**: por distrito (`neighbourhood_group`) y barrio (`neighbourhood`).
6. **Análisis de precios**: precio medio y mediano por barrio y tipo de alquiler.
7. **Actividad reciente y filtrado por fecha (`last_review`)**: análisis de 2025+, 2026+ y anteriores a 2025.
8. **Investigación profunda de anuncios SIN RESEÑAS (`last_review NaN`)**: cruce por anfitrión, tipo de habitación, disponibilidad y licencia.
9. **Auditoría completa de LICENCIAS TURÍSTICAS (`license`)**: total, % con licencia, cruces por tipo de anfitrión, tipo de alojamiento, precio y distrito.

In [ ]:
import pandas as pd
import numpy as np

# 1. Carga del archivo CSV de Inside Airbnb
path_airbnb = '../../data/raw/airbnb/insideairbnb_barcelona_2026-06-24_listings.csv'
df_raw = pd.read_csv(path_airbnb)

print(f"=== 1. CANTAIDAD TOTAL INICIAL ===")
print(f"Total de registros iniciales en Airbnb: {len(df_raw):,}")

In [ ]:
# 2. Eliminación de duplicados por contenido
subset_clave = ['name', 'host_id', 'latitude', 'longitude', 'room_type', 'price']
df = df_raw.drop_duplicates(subset=subset_clave, keep='first').copy()

eliminados = len(df_raw) - len(df)
print(f"=== 2. DEPURACIÓN DE DUPLICADOS ===")
print(f"Anuncios duplicados por contenido idéntico eliminados: {eliminados}")
print(f"Total de anuncios únicos tras deduplicación: {len(df):,}")

In [ ]:
# 3. Análisis de anfitriones y concentración de propiedad
# Etiquetar si un anfitrión es Monopropiedad (1 alojamiento) o Multipropiedad (>1 alojamiento)
df['host_count'] = df['host_id'].map(df['host_id'].value_counts())
df['tipo_anfitrion'] = np.where(df['host_count'] == 1, 'Monopropiedad (1)', 'Multipropiedad (>1)')

total_hosts = df['host_id'].nunique()
avg_listings = len(df) / total_hosts

print(f"=== 3. ANÁLISIS DE ANFITRIONES Y PROPIEDAD ===")
print(f"Cantidad total de host_id únicos: {total_hosts:,}")
print(f"Promedio de alojamientos por anfitrión: {avg_listings:.2f}")

print("\n--- Distribución de Anuncios por Tipo de Anfitrión ---")
dist_anf = df['tipo_anfitrion'].value_counts().reset_index()
dist_anf.columns = ['Tipo de Anfitrión', 'Cantidad de Anuncios']
dist_anf['Porcentaje (%)'] = (dist_anf['Cantidad de Anuncios'] / len(df) * 100).round(2)
display(dist_anf)

print("\nTop 10 anfitriones con mayor número de alojamientos en la plataforma:")
top_hosts = df.groupby(['host_id', 'host_name']).size().reset_index(name='num_alojamientos')
display(top_hosts.sort_values(by='num_alojamientos', ascending=False).head(10))

In [ ]:
# 4. Tipos de alojamiento
room_counts = df['room_type'].value_counts()
room_pcts = df['room_type'].value_counts(normalize=True) * 100

df_room = pd.DataFrame({
    'Cantidad': room_counts,
    'Porcentaje (%)': room_pcts.round(2)
})

print(f"=== 4. TIPOS DE ALOJAMIENTO (ROOM TYPE) ===")
display(df_room)

In [ ]:
# 5. Concentración por grupo de barrio (distrito) y barrio
print(f"=== 5. CONCENTRACIÓN GEOGRÁFICA ===")
distrito_summary = df['neighbourhood_group'].value_counts().reset_index()
distrito_summary.columns = ['Distrito (neighbourhood_group)', 'Cantidad de Anuncios']
distrito_summary['Porcentaje (%)'] = (distrito_summary['Cantidad de Anuncios'] / len(df) * 100).round(2)

print("--- Concentración por Distrito ---")
display(distrito_summary)

print("\n--- Top 15 Barrios con más anuncios ---")
barrio_summary = df.groupby(['neighbourhood_group', 'neighbourhood']).size().reset_index(name='Cantidad de Anuncios')
display(barrio_summary.sort_values(by='Cantidad de Anuncios', ascending=False).head(15))

In [ ]:
# 6. Precio medio por barrio
df['price_clean'] = df['price'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)

precio_barrio = df.groupby(['neighbourhood_group', 'neighbourhood'])['price_clean'].agg(['count', 'mean', 'median', 'min', 'max']).reset_index()
precio_barrio.columns = ['Distrito', 'Barrio', 'Anuncios', 'Precio Medio (€)', 'Mediana (€)', 'Mínimo (€)', 'Máximo (€)']
precio_barrio['Precio Medio (€)'] = precio_barrio['Precio Medio (€)'].round(2)
precio_barrio['Mediana (€)'] = precio_barrio['Mediana (€)'].round(2)

print(f"=== 6. PRECIO MEDIO Y MEDIANO POR BARRIO ===")
print("--- Top 15 Barrios con Mayor Precio Medio ---")
display(precio_barrio.sort_values(by='Precio Medio (€)', ascending=False).head(15))

In [ ]:
# 7. Análisis de fecha de última reseña (last_review)
df['last_review_dt'] = pd.to_datetime(df['last_review'], errors='coerce')

rev_antes_2025 = (df['last_review_dt'] < '2025-01-01').sum()
rev_2025_up = (df['last_review_dt'] >= '2025-01-01').sum()
rev_2026_up = (df['last_review_dt'] >= '2026-01-01').sum()
rev_sin_fecha = df['last_review_dt'].isna().sum()
total = len(df)

print(f"=== 7. ACTIVIDAD RECIENTE (LAST REVIEW) ===")
print(f"Anuncios con última reseña ANTES de 2025 (< 2025-01-01): {rev_antes_2025:,} ({(rev_antes_2025/total*100):.2f}%)")
print(f"Anuncios sin ninguna reseña (NaN): {rev_sin_fecha:,} ({(rev_sin_fecha/total*100):.2f}%)")
print(f"Anuncios con última reseña en 2025 o posterior (>= 2025-01-01): {rev_2025_up:,} ({(rev_2025_up/total*100):.2f}%)")
print(f"Anuncios con última reseña en 2026 o posterior (>= 2026-01-01): {rev_2026_up:,} ({(rev_2026_up/total*100):.2f}%)")

## 8. Investigación Profunda de Anuncios SIN RESEÑAS (`last_review` es NaN)

Analizamos los **3.537 anuncios sin reseñas** cruzándolos con:
1. **Tipo de Anfitrión**: Monopropiedad vs Multipropiedad.
2. **Tipo de Alojamiento** (`room_type`).
3. **Disponibilidad Anual** (`availability_365`).
4. **Licencia**: si figura alguna licencia inscrita.

In [ ]:
df_sin_resenas = df[df['last_review_dt'].isna()].copy()
total_sin_resenas = len(df_sin_resenas)

print(f"=== 8. ANÁLISIS DE ANUNCIOS SIN RESEÑAS (Total: {total_sin_resenas:,}) ===")

# a) Cruce por tipo de anfitrión
print("\n8.1. Cruce por Tipo de Anfitrión:")
dist_anf_sr = df_sin_resenas['tipo_anfitrion'].value_counts().reset_index()
dist_anf_sr.columns = ['Tipo de Anfitrión', 'Cantidad']
dist_anf_sr['Porcentaje (%)'] = (dist_anf_sr['Cantidad'] / total_sin_resenas * 100).round(2)
display(dist_anf_sr)

# b) Cruce por tipo de habitación
print("\n8.2. Cruce por Tipo de Alojamiento (room_type):")
dist_room_sr = df_sin_resenas['room_type'].value_counts().reset_index()
dist_room_sr.columns = ['Tipo de Alojamiento', 'Cantidad']
dist_room_sr['Porcentaje (%)'] = (dist_room_sr['Cantidad'] / total_sin_resenas * 100).round(2)
display(dist_room_sr)

# c) Disponibilidad
print("\n8.3. Estadísticas de Disponibilidad Anual (availability_365):")
display(df_sin_resenas['availability_365'].describe().round(2))

# d) Licencia
df_sin_resenas['tiene_licencia'] = df_sin_resenas['license'].notna() & (df_sin_resenas['license'].astype(str).str.strip() != '') & (df_sin_resenas['license'].astype(str).str.lower() != 'nan')
print("\n8.4. Registrados con Licencia:")
dist_lic_sr = df_sin_resenas['tiene_licencia'].value_counts().reset_index()
dist_lic_sr.columns = ['Tiene Licencia Inscrita', 'Cantidad']
dist_lic_sr['Porcentaje (%)'] = (dist_lic_sr['Cantidad'] / total_sin_resenas * 100).round(2)
display(dist_lic_sr)

## 9. Auditoría Completa de Licencias Turísticas (`license`)

Analizamos la presencia de licencias registradas en todo el conjunto de datos y realizamos cruces con:
1. **Tipo de Anfitrión** (Monopropiedad vs Multipropiedad).
2. **Tipo de Alojamiento** (`room_type`).
3. **Nivel de Precio** (Medio y Mediano en €).
4. **Ubicación por Distrito** (`neighbourhood_group`).

In [ ]:
# 9. Auditoría de Licencias
df['tiene_licencia'] = df['license'].notna() & (df['license'].astype(str).str.strip() != '') & (df['license'].astype(str).str.lower() != 'nan')

total_lic = df['tiene_licencia'].sum()
total_no_lic = len(df) - total_lic

print(f"=== 9. AUDITORÍA GENERAL DE LICENCIAS ===")
print(f"Anuncios CON Licencia inscrita: {total_lic:,} ({(total_lic/len(df)*100):.2f}%)")
print(f"Anuncios SIN Licencia inscrita: {total_no_lic:,} ({(total_no_lic/len(df)*100):.2f}%)")

# Cruce 9.1: Licencia por Tipo de Anfitrión
print("\n--- 9.1. Licencia por Tipo de Anfitrión (%) ---")
cruce_anf = pd.crosstab(df['tipo_anfitrion'], df['tiene_licencia'], normalize='index') * 100
cruce_anf.columns = ['Sin Licencia (%)', 'Con Licencia (%)']
display(cruce_anf.round(2))

# Cruce 9.2: Licencia por Tipo de Alojamiento
print("\n--- 9.2. Licencia por Tipo de Alojamiento (%) ---")
cruce_room = pd.crosstab(df['room_type'], df['tiene_licencia'], normalize='index') * 100
cruce_room.columns = ['Sin Licencia (%)', 'Con Licencia (%)']
display(cruce_room.round(2))

# Cruce 9.3: Precio según Licencia
print("\n--- 9.3. Comparativa de Precios según Licencia (€) ---")
precio_lic = df.groupby('tiene_licencia')['price_clean'].agg(['count', 'mean', 'median', 'min', 'max']).reset_index()
precio_lic.columns = ['Tiene Licencia', 'Cantidad', 'Precio Medio (€)', 'Mediana (€)', 'Mínimo (€)', 'Máximo (€)']
display(precio_lic.round(2))

# Cruce 9.4: Licencia por Distrito
print("\n--- 9.4. Licencia por Distrito (%) ---")
cruce_dist = pd.crosstab(df['neighbourhood_group'], df['tiene_licencia'], normalize='index') * 100
cruce_dist.columns = ['Sin Licencia (%)', 'Con Licencia (%)']
display(cruce_dist.sort_values(by='Con Licencia (%)', ascending=False).round(2))